# Example 05 — Geometric Nonlinearity (Hardening FRF)

2-DOF system with geometric (polynomial) stiffness nonlinearity in modal coordinates.
Matches MATLAB `06_twoSprings_geometricNonlinearity.m`: diagonal M, K, D with
`om1=1.13`, `om2=2`, `zt1=1e-3`, `zt2=5e-3`. Four excitation levels with HB
arc-length continuation (`H=7`, `ds_max=0.005`) and NMA backbone. (Krack & Gross 2019)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.base import MechanicalSystem
from nlvib.nonlinearities.elements import polynomial_stiffness
from nlvib.solvers.harmonic_balance import hb_residual, hb_residual_nma
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System setup (matches MATLAB 06_twoSprings_geometricNonlinearity.m) ---
om1 = 1.13       # MATLAB: first modal frequency
om2 = 2.0        # MATLAB: second modal frequency
zt1 = 1e-3       # MATLAB: first modal damping ratio
zt2 = 5e-3       # MATLAB: second modal damping ratio

M_mat = np.diag([1.0, 1.0])
K_mat = np.diag([om1**2, om2**2])
D_mat = np.diag([2 * zt1 * om1, 2 * zt2 * om2])

OMEGA_LOW   = 0.8
OMEGA_HIGH  = 1.6
N_HARMONICS = 7                            # MATLAB: H=7
EXC_LEVELS  = [3e-4, 5e-4, 1e-3, 3e-3]   # MATLAB: exc_lev
Fex1_dir    = np.array([1.0, 1.0])        # MATLAB: Fex1=[1;1]

def _build_system():
    sys_ = MechanicalSystem(M_mat, D_mat, K_mat)
    # DOF 0: fnl0 = 3*om1^2/2*q0^2 + om1^2/2*q1^2 + om2^2*q0*q1
    #              + (om1^2+om2^2)/2*q0^3 + (om1^2+om2^2)/2*q0*q1^2
    _exp0   = np.array([[2,0],[0,2],[1,1],[3,0],[1,2]], dtype=np.intp)
    _coeff0 = np.array([3*om1**2/2, om1**2/2, om2**2,
                        (om1**2+om2**2)/2, (om1**2+om2**2)/2])
    sys_.add_nonlinear_element(polynomial_stiffness(_exp0, _coeff0, np.array([0,1], dtype=np.intp)))
    # DOF 1: fnl1 = 3*om2^2/2*q1^2 + om2^2/2*q0^2 + om1^2*q0*q1
    #              + (om1^2+om2^2)/2*q1^3 + (om1^2+om2^2)/2*q0^2*q1
    _exp1   = np.array([[2,0],[0,2],[1,1],[3,0],[1,2]], dtype=np.intp)
    _coeff1 = np.array([3*om2**2/2, om2**2/2, om1**2,
                        (om1**2+om2**2)/2, (om1**2+om2**2)/2])
    sys_.add_nonlinear_element(polynomial_stiffness(_exp1, _coeff1, np.array([1,0], dtype=np.intp)))
    return sys_

system  = _build_system()
n_dof   = system.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)

def _make_fext(exc_amp: float) -> np.ndarray:
    """Full HB excitation vector: both DOFs at harmonic 1 cosine block."""
    F = np.zeros(n_total, dtype=np.float64)
    cosine_block = (2 * 1 - 1) * n_dof   # harmonic 1 cosine block index
    F[cosine_block + 0] = exc_amp         # DOF 0
    F[cosine_block + 1] = exc_amp         # DOF 1
    return F

print(f'n_dof={n_dof}, H={N_HARMONICS}, n_total={n_total}')
print(f'om1={om1}, om2={om2}, zt1={zt1}, zt2={zt2}')

In [ ]:
# --- HB FRF arc-length continuation (4 excitation levels, ds_max=0.005) ---
def _run_frf(exc_amp: float) -> tuple:
    """Run HB arc-length continuation for one excitation level.

    Returns (omegas, amp_dof0): fundamental-harmonic amplitude of DOF 0
    = sqrt(cos1^2 + sin1^2). Matches MATLAB: ds=0.005, sweep from 0.8 to 1.6.
    """
    F_ext = _make_fext(exc_amp)

    def residual_fn(Q, omega):
        return hb_residual(Q, omega, system, N_HARMONICS, F_ext)

    # Linear initial guess at omega_start (MATLAB: Q1 = (-Om^2*M + 1i*Om*D + K)\Fex)
    omega_start = OMEGA_LOW
    Fex_vec     = Fex1_dir * exc_amp
    Q1_complex  = np.linalg.solve(
        -(omega_start**2) * M_mat + 1j * omega_start * D_mat + K_mat, Fex_vec
    )
    Q0 = np.zeros(n_total, dtype=np.float64)
    Q0[n_dof * 1 + 0] =  float(np.real(Q1_complex[0]))   # cos1, DOF 0
    Q0[n_dof * 2 + 0] = -float(np.imag(Q1_complex[0]))   # sin1, DOF 0
    Q0[n_dof * 1 + 1] =  float(np.real(Q1_complex[1]))   # cos1, DOF 1
    Q0[n_dof * 2 + 1] = -float(np.imag(Q1_complex[1]))   # sin1, DOF 1

    for _ in range(30):
        R, J = residual_fn(Q0, omega_start)
        if np.linalg.norm(R) < 1e-10: break
        try:    Q0 += np.linalg.solve(J, -R)
        except: Q0 += np.linalg.lstsq(J, -R, rcond=None)[0]

    opts = ContinuationOptions(
        verbose=False,
        ds_initial=0.005,
        ds_min=1e-7,
        ds_max=0.005,          # MATLAB: ds=0.005 (fixed step) — peak error < 1%
        max_steps=5000,
        newton_tol=1e-8,
        max_newton_iter=25,
        lambda_min=None,       # no lower limit — arc-length traces around fold
        lambda_max=OMEGA_HIGH + 0.05,
    )
    result = ContinuationSolver().run(residual_fn, Q0, omega_start, opts)
    print(f'  exc={exc_amp:.0e}: {result.n_steps} steps, {result.message}')

    sol    = result.solutions
    omegas = sol[:, -1]
    cos1   = sol[:, n_dof * 1 + 0]
    sin1   = sol[:, n_dof * 2 + 0]
    amp    = np.sqrt(cos1**2 + sin1**2)
    return omegas, amp

print('Running HB FRF continuation for 4 excitation levels (ds_max=0.005)...')
results_om  = []
results_amp = []
for exc in EXC_LEVELS:
    om_arr, amp_arr = _run_frf(exc)
    results_om.append(om_arr)
    results_amp.append(amp_arr)
print('Done.')

In [ ]:
# --- NMA backbone via HB omega-continuation ---
nma_system = _build_system()  # separate instance (same parameters, D retained)

def nma_res_omega(Q, omega):
    """NMA residual with omega as continuation parameter."""
    z = np.append(Q, omega)
    R_aug, J_aug = hb_residual_nma(z, nma_system, N_HARMONICS)
    R = R_aug[:n_total]
    J = J_aug[:n_total, :n_total]
    return R, J

# Initial guess: near-linear mode 1 at small amplitude
A_SMALL    = 1e-8
omega0_nma = float(np.sqrt(K_mat[0, 0]))   # = om1

Q0_nma = np.zeros(n_total, dtype=np.float64)
Q0_nma[n_dof * 2 + 0] = A_SMALL   # sin1 block, DOF 0

for _it in range(50):
    R0, J0 = nma_res_omega(Q0_nma, omega0_nma)
    if np.linalg.norm(R0) < 1e-12: break
    try:    Q0_nma += np.linalg.solve(J0, -R0)
    except: Q0_nma += np.linalg.lstsq(J0, -R0, rcond=None)[0]

print(f'NMA initial residual: {np.linalg.norm(nma_res_omega(Q0_nma, omega0_nma)[0]):.3e}')

opts_nma = ContinuationOptions(
    verbose=False,
    ds_initial=0.005,
    ds_min=1e-8,
    ds_max=0.05,
    max_steps=500,
    newton_tol=1e-8,
    max_newton_iter=25,
    lambda_min=om1 * 0.95,
    lambda_max=om1 * 1.08,
)
result_nma = ContinuationSolver().run(nma_res_omega, Q0_nma, omega0_nma, opts_nma)
print(f'NMA continuation: {result_nma.n_steps} steps, {result_nma.message}')

sol_nma      = result_nma.solutions
omega_nma_py = sol_nma[:, -1]
cos1_nma     = sol_nma[:, n_dof * 1 + 0]
sin1_nma     = sol_nma[:, n_dof * 2 + 0]
amp_nma_py   = np.sqrt(cos1_nma**2 + sin1_nma**2)
valid_nma    = (omega_nma_py > 0) & np.isfinite(amp_nma_py)
print(f'Valid NMA pts: {valid_nma.sum()}')

In [ ]:
# --- FRF + backbone overlay plot ---
# MATLAB axis limits: xlim=[1.085, 1.15], ylim=[0, 0.35], linear scale
COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

output_dir = Path('..') / 'examples' / '05_geometric_nonlinearity' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

# 4 FRF levels
for i, (om_arr, amp_arr) in enumerate(zip(results_om, results_amp)):
    mask = (om_arr >= 1.085) & (om_arr <= 1.15)
    ax.plot(om_arr[mask], amp_arr[mask],
            color=COLORS[i], linewidth=1.5, label=f'exc={EXC_LEVELS[i]:.0e}')

# NMA backbone
mask_nma = valid_nma & (omega_nma_py >= 1.085) & (omega_nma_py <= 1.15)
ax.plot(omega_nma_py[mask_nma], amp_nma_py[mask_nma],
        'k-', linewidth=2.0, label='NMA backbone')

# Linear natural frequency
ax.axvline(om1, color='k', linestyle='--', linewidth=0.8, alpha=0.7)

ax.set_xlim(1.085, 1.15)   # MATLAB: set(gca,'xlim',[1.085 1.15])
ax.set_ylim(0, 0.35)       # MATLAB: set(gca,'ylim',[0 .35])
ax.set_xlabel('Excitation frequency (rad/s)')
ax.set_ylabel(r'$|Q_{0,1}|$ (fund. harmonic amplitude)')
ax.set_title('Example 05 — Geometric Nonlinearity (hardening FRF + NMA backbone)')
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(output_dir / 'frf.png', dpi=150)
print('Saved frf.png')

In [ ]:
# --- Display FRF inline ---
fig

In [ ]:
# --- Summary ---
print('=' * 60)
print('  Example 05 — Geometric Nonlinearity Summary')
print('=' * 60)
print(f'  System: om1={om1}, om2={om2}, zt1={zt1}, zt2={zt2}')
print(f'  H={N_HARMONICS}, ds_max=0.005, omega=[{OMEGA_LOW}, {OMEGA_HIGH}]')
print()
print(f'  {"exc_lev":<12} {"peak amp (DOF 0, fund)":>24} {"peak omega":>14}')
print('  ' + '-' * 54)
for i, exc in enumerate(EXC_LEVELS):
    mask_i = (results_om[i] >= OMEGA_LOW) & (results_om[i] <= OMEGA_HIGH)
    if mask_i.sum() > 0:
        pk_idx   = int(np.argmax(results_amp[i][mask_i]))
        pk_amp   = float(results_amp[i][mask_i][pk_idx])
        pk_omega = float(results_om[i][mask_i][pk_idx])
        print(f'  {exc:<12.0e} {pk_amp:>24.6f} {pk_omega:>14.4f} rad/s')
    else:
        print(f'  {exc:<12.0e} {"no data":>24}')
print()
if valid_nma.sum() > 0:
    print(f'  NMA backbone: {valid_nma.sum()} pts, '
          f'omega=[{omega_nma_py[valid_nma].min():.4f}, {omega_nma_py[valid_nma].max():.4f}]')
print('=' * 60)